# Laboratorio SQL

Consultas exploratorias sobre el histórico de Supabase. **Es de solo lectura:** cada consulta corre dentro de una transacción `READ ONLY`, así que un `UPDATE` o un `DELETE` escrito acá falla en vez de tocar la base.

No es parte de la corrida diaria. Para el reporte está `pipeline.py`; este notebook solo lee lo que la corrida ya guardó.

In [ ]:
# Como en el notebook principal: subir a la raíz del proyecto (donde está pipeline.py)
import os
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "pipeline.py").exists():
    RAIZ = RAIZ.parent
if not (RAIZ / "pipeline.py").exists():
    raise RuntimeError(f"No encuentro pipeline.py desde {Path.cwd()}")
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from sqlalchemy import text

import data_access

engine = data_access.crear_engine()


def consulta(sql: str, **params) -> pd.DataFrame:
    """Corre `sql` en una transacción de solo lectura y devuelve un DataFrame."""
    with engine.connect() as conn:
        conn.execute(text("SET TRANSACTION READ ONLY"))
        return pd.read_sql_query(text(sql), conn, params=params)


consulta('SELECT COUNT(*) AS filas, MIN("Fecha") AS desde, MAX("Fecha") AS hasta FROM "Fact_Mercado_Macro"')

## Días hábiles sin fila

Lunes a viernes de los últimos 90 días que no tienen fila. Incluye los feriados (la tabla no sabe cuáles son), así que hay que mirar la lista con el calendario al lado: lo que no sea feriado es un día en que la corrida no se hizo.

In [ ]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes"]
faltantes = consulta('''
    SELECT d::date AS fecha, extract(isodow FROM d)::int AS dia_semana
    FROM generate_series(CURRENT_DATE - 90, CURRENT_DATE - 1, interval '1 day') AS d
    WHERE extract(isodow FROM d) < 6
      AND NOT EXISTS (SELECT 1 FROM "Fact_Mercado_Macro" f WHERE f."Fecha" = d::date)
    ORDER BY d
''')
faltantes.assign(dia=faltantes["dia_semana"].map(lambda n: DIAS[n - 1])).drop(columns="dia_semana")

## Brecha entre el blue y el MEP, por mes

Promedio, mínimo y máximo de la brecha diaria. Positiva: el blue está más caro que el MEP.

In [ ]:
brecha = consulta('''
    SELECT date_trunc('month', "Fecha")::date AS mes,
           ROUND(AVG(("TCV_Blue" / "TCV_MEP" - 1) * 100)::numeric, 2) AS brecha_promedio,
           ROUND(MIN(("TCV_Blue" / "TCV_MEP" - 1) * 100)::numeric, 2) AS brecha_minima,
           ROUND(MAX(("TCV_Blue" / "TCV_MEP" - 1) * 100)::numeric, 2) AS brecha_maxima,
           COUNT(*) AS ruedas
    FROM "Fact_Mercado_Macro"
    WHERE "Fecha" >= :desde AND "TCV_MEP" > 0
    GROUP BY 1
    ORDER BY 1
''', desde="2025-01-01")
brecha

In [ ]:
brecha.plot(x="mes", y=["brecha_promedio", "brecha_minima", "brecha_maxima"], figsize=(10, 4),
            title="Brecha blue contra MEP (%)", grid=True);

## Riesgo país por mes

In [ ]:
consulta('''
    SELECT date_trunc('month', "Fecha")::date AS mes,
           MIN(riesgo_pais) AS minimo, MAX(riesgo_pais) AS maximo,
           ROUND(AVG(riesgo_pais)::numeric, 0) AS promedio,
           (array_agg(riesgo_pais ORDER BY "Fecha" DESC))[1] AS cierre
    FROM "Fact_Mercado_Macro"
    WHERE "Fecha" >= :desde
    GROUP BY 1
    ORDER BY 1
''', desde="2025-01-01")

## Variación mensual de blue, MEP y billete

Último dato de cada mes contra el último del mes anterior.

In [ ]:
cierres = consulta('''
    SELECT DISTINCT ON (date_trunc('month', "Fecha"))
           date_trunc('month', "Fecha")::date AS mes, "TCV_Blue", "TCV_MEP", "TCV_Billete"
    FROM "Fact_Mercado_Macro"
    WHERE "Fecha" >= :desde
    ORDER BY date_trunc('month', "Fecha"), "Fecha" DESC
''', desde="2024-12-01").set_index("mes")
(cierres.pct_change() * 100).round(2).dropna()

## Párrafos de IA: qué modelo los escribió

Cuántos párrafos salieron de cada modelo y cuántos días quedaron sin párrafo (`ai_paragraph` vacío o nulo).

In [ ]:
consulta('''
    SELECT COALESCE(ai_model, '(sin modelo)') AS modelo,
           COUNT(*) AS dias,
           COUNT(*) FILTER (WHERE COALESCE(ai_paragraph, '') = '') AS sin_parrafo,
           ROUND(AVG(length(ai_paragraph))) AS largo_promedio,
           MIN("Fecha") AS primero, MAX("Fecha") AS ultimo
    FROM "Fact_Mercado_Macro"
    WHERE "Fecha" >= :desde
    GROUP BY 1
    ORDER BY dias DESC
''', desde="2026-01-01")

## Series monetarias (fase 3)

Solo si ya se aplicó `sql/06_series_macro.sql`. Muestra cada serie con su unidad y su último dato.

In [ ]:
if data_access.tabla_existe(engine):
    display(consulta('''
        SELECT serie, frecuencia, unidad, COUNT(*) AS puntos, MIN("Fecha") AS desde, MAX("Fecha") AS hasta,
               (array_agg(valor ORDER BY "Fecha" DESC))[1] AS ultimo_valor
        FROM "Fact_Series_Macro"
        GROUP BY serie, frecuencia, unidad
        ORDER BY serie
    '''))
else:
    print("Todavía no existe Fact_Series_Macro: ver docs/fase-3-agregados-y-deuda.md")

## Consulta libre

Escribí acá lo que quieras explorar. Recordá que la transacción es de solo lectura.

In [ ]:
consulta('''
    SELECT *
    FROM "Fact_Mercado_Macro"
    ORDER BY "Fecha" DESC
    LIMIT 10
''')